In [72]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , PydanticOutputParser 
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

from langchain.schema.runnable import RunnableSequence , RunnableBranch , RunnablePassthrough
from langchain.tools import BaseTool 
from typing import Type
from pydantic import BaseModel , Field
from langchain.memory import ChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory



load_dotenv()

llm = HuggingFaceEndpoint(
    # repo_id = "google/gemma-2-2b-it",
    repo_id="deepseek-ai/DeepSeek-R1-0528",

    # huggingfacehub_api_token="YOUR_HF_TOKEN",
    huggingfacehub_api_token="YOUR_HF_TOKEN",
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)


In [73]:
claim = "Albert einstein has more accomplishments then isaac newton"

In [74]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=10,
    topic="general",
    tavily_api_key = "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"
  
)

In [75]:
result = tool.invoke({"query" : claim})

In [76]:
result

{'query': 'Albert einstein has more accomplishments then isaac newton',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.quora.com/Who-had-more-academic-achievements-and-notable-scientific-contributions-Albert-Einstein-or-Sir-Isaac-Newton',
   'title': 'Who had more academic achievements and notable scientific ...',
   'content': 'Newton did more for the advancement of science than Einstein did. Basic gravity, motion, light, optics, mathematics - just some of the areas',
   'score': 0.71088976,
   'raw_content': None},
  {'url': 'https://www.thepromptmag.com/whos-better-newton-or-einstein-settling-the-debate-once-and-for-all',
   'title': "Who's Better—Newton or Einstein? Settling the Debate Once and ...",
   'content': 'In any case, considering LeBron’s legacy has me thinking about a similar question that you’ll find discussed among the physics phan-boys and phan-girls of the world: *Who was the greatest physicist ever—Isaac Newton or Alb

In [77]:
text =  " ".join(item["content"] for item in result['results'] if item.get("content"))


In [78]:
text

'Newton did more for the advancement of science than Einstein did. Basic gravity, motion, light, optics, mathematics - just some of the areas In any case, considering LeBron’s legacy has me thinking about a similar question that you’ll find discussed among the physics phan-boys and phan-girls of the world: *Who was the greatest physicist ever—Isaac Newton or Albert Einstein?* Newton also apparently had a sudden “lightbulb” moment, much like Einstein would have centuries later. From this idea Newton went on to develop not just a “pillar” of physics, as Einstein did with his theory of relativity, but the *whole damn foundation*. We can’t just compare Einstein’s best thought to Newton’s best thought. This understanding of Einstein’s genius, that it was a genius of originality, rather than one of technical mastery, matters when comparing Einstein and Newton. We particularly focus on and compare two of the greatest physicists,. Isaac Newton and Albert Einstein, depicting their fame evolutio

In [79]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap = 100)
chunks = splitter.create_documents([text])

In [80]:
chunks

[Document(metadata={}, page_content='Newton did more for the advancement of science than Einstein did. Basic gravity, motion, light, optics, mathematics - just some of the areas In any case, considering LeBron’s legacy has me thinking about a similar question that you’ll find discussed among the physics phan-boys and phan-girls of the world: *Who was the greatest physicist ever—Isaac Newton or Albert Einstein?* Newton also apparently had a sudden “lightbulb” moment, much like Einstein would have centuries later. From this idea'),
 Document(metadata={}, page_content='had a sudden “lightbulb” moment, much like Einstein would have centuries later. From this idea Newton went on to develop not just a “pillar” of physics, as Einstein did with his theory of relativity, but the *whole damn foundation*. We can’t just compare Einstein’s best thought to Newton’s best thought. This understanding of Einstein’s genius, that it was a genius of originality, rather than one of technical mastery, matter

In [ ]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings


embedding_model = "sentence-transformers/all-mpnet-base-v2"
hf = HuggingFaceEndpointEmbeddings(
    model=embedding_model,
    task="feature-extraction",
    # huggingfacehub_api_token="YOUR_HF_TOKEN",
    huggingfacehub_api_token="YOUR_HF_TOKEN",
)

In [84]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, hf) 
# chunks are converted to embeddings and stored in vector store

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/hf-inference/models/sentence-transformers/all-mpnet-base-v2/pipeline/feature-extraction (Request ID: Root=1-68ef2579-465531c23ab510843e9fbf05;6eae5f89-4d92-42b8-b902-d6df6d1aea30)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
retriever = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = { "k" : 5 , "lambda_mult" : 0.9}
)

In [ ]:
docs = retriever.invoke(claim)

In [ ]:
summary_prompt = PromptTemplate(
    template = """Summarize the key facts from the following search results relevant to this claim,
    claim : {claim} ,
    documents : {docs}
    Return a concise factual summary.
    """,
    input_variables=['claim' , 'docs']
)


In [ ]:
parser = StrOutputParser()
summary_chain = summary_prompt | model | parser

In [ ]:
summary = summary_chain.invoke({'claim' : claim , 'docs' : docs })

In [ ]:
summary

'While people frequently debate who is "better" - Isaac Newton or Albert Einstein - the documents present a clear distinction between their contributions:\n\n* **Newton**: Is considered the "founder" of physics, solidifying laws of motion and universal gravitation. \n* **Einstein**: Developed groundbreaking theories specifically regarding the theory of relativity and E=MC².  \n\nInterpretations and timelines may influence opinions. A core difference becomes apparent: **Newton built essential foundations, while Einstein revolutionized the *nature* of physics with entirely new theories.** Both are considered giants in their own right. \n'

In [ ]:
# from pydantic import BaseModel , Field
# from enum import Enum
# class Verdict(Enum):
#     TRUE = 'True'
#     FALSE = 'False'
#     UNCERTAIN = 'Uncertain'

# class Classify(BaseModel):
#     verdict : Verdict  = Field(description = "The final verdict of the claim : 'True' , 'False' , 'Uncertain'")

response_schemas = [
    ResponseSchema(name="verdict", description="Final decision: True, False, or Uncertain"),
    ResponseSchema(name="reasoning", description="Reasoning behind the verdict")
]

structuredoutput_parser = StructuredOutputParser.from_response_schemas(response_schemas)


In [ ]:
format_instructions = structuredoutput_parser.get_format_instructions()


In [ ]:
verify_prompt = PromptTemplate(
template = """
You are a fact-checking analyst.

Based on the evidence below, classify the claim as:
- TRUE
- FALSE
- UNCERTAIN (if insufficient or conflicting evidence)

Claim: {claim}
Evidence Summary: {summary}

Return your verdict and short reasoning.
{format_instruction}
""",
input_variables = ['claim' , 'summary'],
partial_variables={'format_instruction' : format_instructions}
)



In [ ]:
verify_chain = verify_prompt | model | structuredoutput_parser

In [ ]:
verify_result = verify_chain.invoke({'claim' : claim , 'summary' : summary})

In [ ]:
verify_result


{'verdict': 'UNCERTAIN',
 'reasoning': 'While the evidence supports that Einstein contributed groundbreaking theories that revolutionized physics, it does not provide a direct comparative assessment of total accomplishments.  The text highlights the different contributions of each, but does not specify which individual has more overall achievements across their respective fields.'}

In [ ]:
verify_result['verdict']

'UNCERTAIN'

In [ ]:

response_schemas = [
    ResponseSchema(name="Confidence_Rating", description="Confidence that the verdict is correct"),
    ResponseSchema(name="reasoning", description="Reasoning behind the verdict and its confidence rating")
]

structuredoutput_parser = StructuredOutputParser.from_response_schemas(response_schemas)


In [ ]:
eval_prompt = PromptTemplate(
    template = """
 Evaluate the following verdict:
 Verdict: {verdict}
 Reasoning: {reasoning}
Rate confidence from 0 (very uncertain) to 1 (very confident).
If confidence < 0.6, suggest if retrieval should be expanded or rechecked
{format_instruction}
                             
""" ,
input_variables = ['verdict' , 'reasoning'],
partial_variables={'format_instruction' : structuredoutput_parser.get_format_instructions()})

In [ ]:
eval_chain = eval_prompt | model | structuredoutput_parser

In [ ]:
eval_result = eval_chain.invoke({'verdict' : verify_result['verdict'] , 'reasoning' : verify_result['reasoning']})

In [ ]:
eval_result

{'Confidence_Rating': '0.5',
 'reasoning': "The verdict is somewhat uncertain. While it acknowledges Einstein's groundbreaking contributions, it points out the lack of direct comparison of individual achievements across fields.  Without a comparative assessment, it's impossible to conclusively determine which individual has more total accomplishments. A broader retrieval or review may be needed to obtain more complete information for comparison and a more precise verdict."}

In [ ]:

response_schemas = [
    ResponseSchema(name="Confidence_Rating", description="Confidence that the verdict is correct"),
    ResponseSchema(name="reasoning", description="Reasoning behind the verdict and its confidence rating")
]


structuredoutput_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [ ]:
re_eval_prompt = PromptTemplate(
    template = """The confidence rating on the claim was {confidence_rating}
    Reasoning for the verdict was {reasoning},
    Re-evaluate the confidence rating according to the reasoning , claim and verdict.
    {format_instruction}
    """,
    input_variables=['confidence_rating' , 'reasoning'],
    partial_variables = {'format_instruction' : structuredoutput_parser.get_format_instructions()})


In [ ]:

re_eval_chain = re_eval_prompt | model | structuredoutput_parser
re_evaluation = re_eval_chain.invoke({'confidence_rating' : eval_result['Confidence_Rating'] , 'reasoning' : eval_result['reasoning']})

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/nebius/v1/chat/completions (Request ID: Root=1-68ef24e9-10f07b66794edc9c466aefad;13aae251-dfb7-40aa-91e1-3f84efc929c6)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
re_evaluation

'Given the reasoning presented, the confidence rating should be **lowered** from 0.6 to **0.4 or 0.3**. \n\nHere\'s why:\n\n* **Ambiguous Verdict:** The verdict highlights a lack of clear comparison metrics, implying it\'s not a definitive judgment about "who\'s more accomplished." This is crucial because a confidence rating of 0.6 suggests a strong level of certainty, which is inconsistent with the verdict\'s uncertainty. \n* **Need For Further Information:** The reasoning stresses the importance of considering a broader range of contributions and individual achievements.  This means the current verdict lacks solid, verifiable evidence to claim a definitive “more accomplished” outcome. \n* **Probabilistic Judgement:** A confidence rating lower than 0.6 points to a judgment that isn\'t as confident, and acknowledges the need for further information to make the comparison more concrete.\n\n**Recommendations:**\n\n* Acknowledge the inconclusive nature of the verdict.\n* Clearly state tha

In [ ]:
self_correct_branch = RunnableBranch(
    (lambda x : x['Confidence_Rating'] >= 0.6 , RunnablePassthrough()),
    
)

ValueError: RunnableBranch requires at least two branches